In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# def df_to_dataset(dataframe, shuffle=True, batch_size=32):
#   dataframe = dataframe.copy()
#   labels = dataframe.pop('expenses')
#   ds = tf.data.Dataset.from_tensor_slices((dict(dataframe), labels))
#   if shuffle:
#     ds = ds.shuffle(buffer_size=len(dataframe))
#   ds = ds.batch(batch_size)
#   return ds

_train_dataset, _test_dataset = train_test_split(dataset, test_size=0.2,random_state=42)
_train_dataset, _eval_dataset = train_test_split(_train_dataset, test_size=0.2, random_state=42)
train_labels = np.log(_train_dataset.pop('expenses').values.astype(np.float32))
test_labels = np.log(_test_dataset.pop('expenses').values.astype(np.float32))
eval_labels = np.log(_eval_dataset.pop('expenses').values.astype(np.float32))

STRING_COLS = ['sex', 'smoker', 'region']
NUMERIC_COLS = ['age', 'bmi', 'children']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(feature_range=(0, 1)), NUMERIC_COLS),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), STRING_COLS)
    ])

train_dataset = preprocessor.fit_transform(_train_dataset).astype(np.float32)
eval_dataset = preprocessor.transform(_eval_dataset).astype(np.float32)
test_dataset = preprocessor.transform(_test_dataset).astype(np.float32)
input_dim = train_dataset.shape[1]


In [ ]:
# def input_fn(features, labels):
#   ds = tf.data.Dataset.from_tensor_slices((dict(features), labels))
#   ds.shuffle(1000).repeat()
#   return ds.batch(32)

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)
])
model.compile(loss="mse",optimizer=keras.optimizers.Adam(learning_rate=0.001), metrics=['mae', 'mse'])
model.fit(train_dataset, train_labels, validation_data=(eval_dataset, eval_labels), epochs=250, batch_size=128)


In [ ]:
# reset the test values to be the proper scale

test_labels = np.exp(test_labels)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

# have to modify the values to be proper, after passing in logs to train earleir
predictions_log = model.predict(test_dataset).flatten()
predictions_dollars = np.exp(predictions_log)
actual_mae = np.mean(np.abs(predictions_dollars - test_labels))
mae=actual_mae

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

# again, modify
test_predictions = np.exp(test_predictions)

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
